## EIA Brent-WTI Prices
#### Coverage: 2026 - 2017

In [13]:
from dotenv import load_dotenv
import requests
import pandas as pd
import os

load_dotenv()

True

In [14]:
EIA_API_KEY = os.getenv('EIA_API_KEY')
FRED_API_KEY = os.getenv('FRED_API_KEY')
START_DATE = os.getenv('START_DATE')

headers = {
    "X-Params": f"api_key={EIA_API_KEY}"
}

### Brent Crude Prices

In [15]:
series_id = 'RBRTE'
price_name = 'Brent_Price_USD'

In [16]:
series_id = 'RBRTE'
price_name = 'Brent_Price_USD'

# Build URL with api_key as a query parameter
url = (
    f"https://api.eia.gov/v2/petroleum/pri/spt/data/"
    f"?api_key={EIA_API_KEY}"
    f"&frequency=daily"
    f"&data[0]=value"
    f"&facets[series][]={series_id}"
    f"&start={START_DATE}"
    f"&sort[0][column]=period"
    f"&sort[0][direction]=desc"
    f"&length=5000"
)

In [17]:
series_id = 'RBRTE'
price_name = 'Brent_Price_USD'

# Build URL with api_key as a query parameter
url = (
    f"https://api.eia.gov/v2/petroleum/pri/spt/data/"
    f"?api_key={EIA_API_KEY}"
    f"&frequency=daily"
    f"&data[0]=value"
    f"&facets[series][]={series_id}"
    f"&start={START_DATE}"
    f"&sort[0][column]=period"
    f"&sort[0][direction]=desc"
    f"&length=5000"
)

In [18]:
response = requests.get(url)

if response.status_code == 200:
    data = response.json()['response']['data']
    df_brent = pd.DataFrame(data)
    
    # Process Date and value columns as per utils.py logic
    df_brent = df_brent[['period', 'value']]
    df_brent.rename(columns={'period': 'Date', 'value': price_name}, inplace=True)
    df_brent['Date'] = pd.to_datetime(df_brent['Date']).dt.tz_localize(None)
    
    print(f"Fetched {len(df_brent)} Brent records.")
    display(df_brent.head())
else:
    print(f"Failed Brent fetch: {response.status_code} - {response.text}")

Fetched 2363 Brent records.


,Date,Brent_Price_USD
0,2026-04-27,113.89
1,2026-04-24,111.86
2,2026-04-23,113.25
3,2026-04-22,113.44
4,2026-04-21,106.14


### WTI Crude Prices

In [19]:
series_id = 'RWTC'
price_name = 'WTI_Price_USD'

In [20]:
url = (
    f"https://api.eia.gov/v2/petroleum/pri/spt/data/"
    f"?api_key={EIA_API_KEY}"
    f"&frequency=daily"
    f"&data[0]=value"
    f"&facets[series][]={series_id}"
    f"&start={START_DATE}"
    f"&sort[0][column]=period"
    f"&sort[0][direction]=desc"
    f"&length=5000"
)

In [21]:
response = requests.get(url)

if response.status_code == 200:
    data = response.json()['response']['data']
    df_wti = pd.DataFrame(data)
    
    df_wti = df_wti[['period', 'value']]
    df_wti.rename(columns={'period': 'Date', 'value': price_name}, inplace=True)
    df_wti['Date'] = pd.to_datetime(df_wti['Date']).dt.tz_localize(None)
    
    print(f"Fetched {len(df_wti)} WTI records.")
    display(df_wti.head())

Fetched 2331 WTI records.


,Date,WTI_Price_USD
0,2026-04-27,99.89
1,2026-04-24,98.42
2,2026-04-23,99.27
3,2026-04-22,94.76
4,2026-04-21,93.64


# FRED Observations Endpoint


In [25]:
url = "https://api.stlouisfed.org/fred/series/observations"

In [26]:
series_id = 'POILDUBUSDM'  # Dubai Crude Global Price
price_name = 'Dubai_Crude_USD'

In [27]:
params = {
    'series_id': series_id,
    'api_key': FRED_API_KEY,
    'file_type': 'json',
    'observation_start': START_DATE,
    'sort_order': 'desc'
}

In [28]:
response = requests.get(url, params=params)

if response.status_code == 200:
    data = response.json()
    observations = data.get('observations', [])
    
    df_dubai = pd.DataFrame(observations)
    
    # Keep only the date and value columns
    df_dubai = df_dubai[['date', 'value']]
    df_dubai.rename(columns={'date': 'Date', 'value': price_name}, inplace=True)
    
    # 1. Convert Date to Datetime
    df_dubai['Date'] = pd.to_datetime(df_dubai['Date'])
    
    # 2. Convert Value to Numeric. 
    # FRED uses '.' for missing values; 'coerce' turns them into actual NaNs
    df_dubai[price_name] = pd.to_numeric(df_dubai[price_name], errors='coerce')
    
    # 3. Drop rows with NaN (missing prices)
    df_dubai.dropna(subset=[price_name], inplace=True)
    
    # Reset index after dropping rows
    df_dubai.reset_index(drop=True, inplace=True)
    
    print(f"Fetched and cleaned {len(df_dubai)} records for Dubai Crude.")
    display(df_dubai.head())
else:
    print(f"Failed fetch: {response.status_code} - {response.text}")

Fetched and cleaned 111 records for Dubai Crude.


,Date,Dubai_Crude_USD
0,2026-03-01,126.706364
1,2026-02-01,68.508500
2,2026-01-01,62.728182
3,2025-12-01,61.830000
4,2025-11-01,64.355500
